## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-12-14-54-47 +0000,bbc,Ukraine says military Starlink disrupted again...,https://www.bbc.co.uk/news/articles/cwrkgmymk5...
1,2026-11-12-14-54-00 +0000,wsj,"Multiple People Shot in Washington, D.C.",https://www.wsj.com/us-news/multiple-people-sh...
2,2026-11-12-14-52-05 +0000,startribune,Nvidia CEO Jensen Huang resigns after his affa...,https://www.startribune.com/megaspeed-firm-cen...
3,2026-11-12-14-52-01 +0000,nypost,"Brooklyn man, 26, charged with drawing Nazi sw...",https://nypost.com/2026/11/12/us-news/brooklyn...
4,2026-11-12-14-51-28 +0000,nyt,Nvidia Stock Plunges After Jensen Huang Resign...,https://www.nytimes.com/2026/11/12/business/nv...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2674/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
105,trump,31
11,nvidia,31
14,huang,28
13,jensen,26
35,new,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
20,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...,177
268,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...,167
51,2026-11-12-14-00-00 +0000,wsj,President Trump said the U.S. doesn’t need a d...,https://www.wsj.com/politics/policy/trump-now-...,145
397,2026-11-11-15-04-13 +0000,nypost,Nvidia CEO Jensen Huang tried to act ‘business...,https://nypost.com/2026/11/11/business/nvidias...,138
206,2026-11-12-02-00-00 +0000,wsj,Democratic Sens. Elizabeth Warren and Chris Co...,https://www.wsj.com/politics/policy/top-democr...,136


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
20,177,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...
51,114,2026-11-12-14-00-00 +0000,wsj,President Trump said the U.S. doesn’t need a d...,https://www.wsj.com/politics/policy/trump-now-...
191,99,2026-11-12-04-02-00 +0000,wsj,Democrat Scott Colom is the state of Mississip...,https://www.wsj.com/politics/elections/scott-c...
228,71,2026-11-12-00-24-23 +0000,chitri,Chicago parents and students protest Google’s ...,https://www.chicagotribune.com/2026/11/11/cps-...
165,65,2026-11-12-08-24-15 +0000,nypost,Texas cruise passenger Susan Watford’s adoptiv...,https://nypost.com/2026/11/12/us-news/susan-wa...
205,60,2026-11-12-02-00-00 +0000,wsj,“The ass-kicking we needed.” South Carolina Re...,https://www.wsj.com/politics/elections/republi...
125,57,2026-11-12-11-00-00 +0000,latimes,"L.A. sales taxes will go up, but the time it t...",https://www.latimes.com/california/story/2026-...
189,55,2026-11-12-04-07-07 +0000,nypost,Democratic House lawmaker gets emotional as sh...,https://nypost.com/2026/11/11/us-news/democrat...
351,45,2026-11-11-17-58-00 +0000,wsj,Spain Hit by General Strike as Cost of Living ...,https://www.wsj.com/world/europe/spain-hit-by-...
258,45,2026-11-11-22-39-00 +0000,wsj,U.S. Treasury Secretary Scott Bessent was rece...,https://www.wsj.com/politics/national-security...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
